# CS781 Capstone -- Phase 5: Legal Statute Identification (LSI)

Multi-label classification -- each document maps to a *set* of statute IDs. All 6 required
experiments, one per technique, four Phase 1 models represented, trained from scratch per your
instructor's requirement, in one notebook.

| # | Model | Technique |
|---|---|---|
| 1 | GQA | LoRMA (compulsory) |
| 2 | GPT-2 | Full Fine-tuning |
| 3 | SwiGLU | AdaLoRA (peft) |
| 4 | Prefix-LM | Adapters (Houlsby bottleneck) |
| 5 | GQA | Prefix Tuning (hand-rolled) |
| 6 | winner of #1-#5 | Knowledge Distillation (mandatory) |



## 1. Hugging Face Login

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

In [ ]:
import os, random, json, time, math, zipfile, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_5_LSI/Train/train.json"
DEV_FILE   = "Phase_5_LSI/Train/dev.json"
TEST_FILE  = "Phase_5_LSI/Test/test.json"

MODEL_PATHS = {
    "gpt2":    "/kaggle/input/datasets/himangshumahato/model1",
    "swiglu":  "/kaggle/input/datasets/himangshumahato/model2",
    "gqa":     "/kaggle/input/datasets/himangshumahato/model3",
    "prefixlm":"/kaggle/input/datasets/himangshumahato/model4",
}
MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD = 768, 12, 12
N_KV_HEAD = 4

HEAD_TOKENS, TAIL_TOKENS = 512, 256
DOC_MAX_LEN = HEAD_TOKENS + TAIL_TOKENS   # 768, matches your validated Phase 4 config

SAMPLES_PER_EPOCH = 12000
DEV_SUBSET_SIZE = 2500

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase5"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {
        "dir": d,
        "model_path": os.path.join(d, "best_model.pt"),
        "run_info_path": os.path.join(d, "run_info.json"),
        "prediction_path": os.path.join(d, "prediction.json"),
        "submission_zip": os.path.join(d, "submission.zip"),
    }

print("Configuration loaded.")

## 4. Install peft (used for AdaLoRA, Experiment 3)

In [ ]:
!pip uninstall -y torchao -q 2>/dev/null
!pip -q install regex peft accelerate
import peft
print("peft version:", peft.__version__)

## 5. Tokenizer (same shared tokenizer as every prior phase)

In [ ]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

In [ ]:
from a1_tokenizer import ByteLevelBPE, _PAT

tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gqa"], "bpe_tokenizer.json"))
print("Vocabulary size:", len(tokenizer))
for name, path in MODEL_PATHS.items():
    t = ByteLevelBPE.load(os.path.join(path, "bpe_tokenizer.json"))
    assert t.vocab_size == tokenizer.vocab_size, f"{name} tokenizer differs!"
print("All four checkpoints share the same tokenizer.")

## 6. Load Phase 5 data

In [ ]:
from huggingface_hub import hf_hub_download

def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

train_raw = hf_json(TRAIN_FILE)
dev_raw   = hf_json(DEV_FILE)
test_raw  = hf_json(TEST_FILE)

print("train:", len(train_raw), type(train_raw))
print("dev:", len(dev_raw), type(dev_raw))
print("test:", len(test_raw), type(test_raw))

sample = train_raw[0] if isinstance(train_raw, list) else next(iter(train_raw.values()))
print("sample:", {k: (str(v)[:120] if k == "text" else v) for k, v in sample.items()} if isinstance(sample, dict) else sample)

## 7. Normalize into records, build the statute vocabulary

In [ ]:
LABEL_KEYS = ["labels", "statutes", "statute_ids", "label", "pred"]
TEXT_KEYS = ["text", "case_text", "document"]

def find_key(example, candidates):
    for k in candidates:
        if k in example:
            return k
    return None

def to_records(raw, with_labels=True):
    items = raw.items() if isinstance(raw, dict) else [(d.get("id", d.get("case_id")), d) for d in raw]
    records = []
    label_key, text_key = None, None
    for doc_id, d in items:
        if isinstance(d, str):
            text, labels = d, None
        else:
            if text_key is None:
                text_key = find_key(d, TEXT_KEYS)
                assert text_key is not None, f"no recognized text key in {list(d.keys())}"
            text = d[text_key]
            if with_labels:
                if label_key is None:
                    label_key = find_key(d, LABEL_KEYS)
                    assert label_key is not None, f"no recognized label key in {list(d.keys())}"
                labels = d.get(label_key, [])
                if not isinstance(labels, list):
                    labels = [labels]
            else:
                labels = None
        rec = {"id": doc_id, "text": text}
        if with_labels:
            rec["labels"] = [int(l) for l in labels]
        records.append(rec)
    return records

train_records = to_records(train_raw, with_labels=True)
dev_records   = to_records(dev_raw,   with_labels=True)
test_records  = to_records(test_raw,  with_labels=False)

print("train:", len(train_records), "| dev:", len(dev_records), "| test:", len(test_records))

STATUTE_SET = sorted({s for r in train_records + dev_records for s in r["labels"]})
STATUTE2ID = {s: i for i, s in enumerate(STATUTE_SET)}
ID2STATUTE = {i: s for s, i in STATUTE2ID.items()}
N_STATUTES = len(STATUTE_SET)
print(f"statute vocabulary size: {N_STATUTES}")
print(f"labels per train doc -- min/mean/max: "
     f"{min(len(r['labels']) for r in train_records)}/"
     f"{np.mean([len(r['labels']) for r in train_records]):.2f}/"
     f"{max(len(r['labels']) for r in train_records)}")

## 8. Document length distribution

In [ ]:
lengths = np.array([len(tokenizer.encode(r["text"])) for r in train_records[:200]])
for p in [50, 75, 90, 95, 99, 100]:
    print(f"p{p:>4}: {np.percentile(lengths, p):.0f} tokens")
over_limit = (lengths > DOC_MAX_LEN).mean()
print(f"fraction of (sampled) documents exceeding {DOC_MAX_LEN} tokens: {over_limit:.1%}")

## 9. Dataset / collator -- head+tail truncation, multi-hot labels

In [ ]:
PAD_ID = tokenizer.eos_token_id

def encode_head_tail(text, head=HEAD_TOKENS, tail=TAIL_TOKENS):
    ids = tokenizer.encode(text)
    if len(ids) <= head + tail:
        return ids
    return ids[:head] + ids[-tail:]


class LSIDataset(torch.utils.data.Dataset):
    def __init__(self, records, with_labels=True):
        self.records = records
        self.with_labels = with_labels

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        if "ids" not in r:                                   # tokenize once, cache across epochs
            r["ids"] = np.array(encode_head_tail(r["text"]), dtype=np.int32)
        item = {"ids": r["ids"], "idx": idx}
        if self.with_labels:
            multihot = np.zeros(N_STATUTES, dtype=np.float32)
            for s in r["labels"]:
                if s in STATUTE2ID:                           # silently drop dev/test-only unseen statutes
                    multihot[STATUTE2ID[s]] = 1.0
            item["label"] = multihot
        return item


def lsi_collator(batch):
    L = DOC_MAX_LEN
    B = len(batch)
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((B, L), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["ids"])
        input_ids[i, :n] = torch.tensor(item["ids"])
        attention_mask[i, :n] = True
    out = {"input_ids": input_ids, "attention_mask": attention_mask,
           "record_idx": torch.tensor([item["idx"] for item in batch])}
    if "label" in batch[0]:
        out["labels"] = torch.tensor(np.stack([item["label"] for item in batch]))
    return out

_t = lsi_collator([LSIDataset(train_records)[i] for i in range(4)])
print({k: v.shape for k, v in _t.items()})

## 10. Per-epoch subsampling (same speed trade-off as your Phase 4 run)

In [ ]:
rng = np.random.RandomState(SEED)
dev_records_small = [dev_records[i] for i in rng.permutation(len(dev_records))[:DEV_SUBSET_SIZE]]
print("dev subset for per-epoch eval:", len(dev_records_small), "/ full dev:", len(dev_records))

def make_train_loader(batch_size):
    ds = LSIDataset(train_records)
    sampler = torch.utils.data.RandomSampler(ds, replacement=False,
                                             num_samples=min(SAMPLES_PER_EPOCH, len(ds)))
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, sampler=sampler, collate_fn=lsi_collator)

def make_dev_loader(batch_size=8):
    return torch.utils.data.DataLoader(LSIDataset(dev_records_small), batch_size=batch_size,
                                       shuffle=False, collate_fn=lsi_collator)

def make_full_dev_loader(batch_size=8):
    return torch.utils.data.DataLoader(LSIDataset(dev_records), batch_size=batch_size,
                                       shuffle=False, collate_fn=lsi_collator)

## 11. Per-statute positive weights (inverse frequency, for BCEWithLogitsLoss)

In [ ]:
pos_counts = np.zeros(N_STATUTES)
for r in train_records:
    for s in r["labels"]:
        if s in STATUTE2ID:
            pos_counts[STATUTE2ID[s]] += 1
n_train = len(train_records)
neg_counts = n_train - pos_counts
pos_weight_arr = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 50.0)   # capped, same philosophy as Phase 3's class-weight clip
pos_weight = torch.tensor(pos_weight_arr, dtype=torch.float).to(device)
print(f"pos_weight: min {pos_weight_arr.min():.1f}, median {np.median(pos_weight_arr):.1f}, max {pos_weight_arr.max():.1f}")
print(f"statutes with <5 train examples: {(pos_counts < 5).sum()} / {N_STATUTES}")

## 12. Backbone architectures

In [ ]:
import math as _m

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, hidden_mult=4):
        super().__init__()
        hidden = int(n_embd * hidden_mult * 2 / 3)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class CausalSelfAttentionRMS(CausalSelfAttention):
    pass


class RMSSwiGLUBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd); self.attn = CausalSelfAttentionRMS(n_embd, n_head, n_positions, dropout)
        self.ln_2 = RMSNorm(n_embd); self.mlp = SwiGLUMLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class RMSSwiGLULM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([RMSSwiGLUBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head; self.num_groups = n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class ScratchGroupedQueryLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

print("Backbone classes defined.")

## 13. Backbone registry and loader

In [ ]:
from safetensors.torch import load_file

BACKBONE_REGISTRY = {
    "gpt2":     {"cls": GPT2ScratchLM,       "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "swiglu":   {"cls": RMSSwiGLULM,         "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "gqa":      {"cls": ScratchGroupedQueryLM, "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
    "prefixlm": {"cls": PrefixLM,            "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
}

def load_backbone(name):
    spec = BACKBONE_REGISTRY[name]
    model = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **spec["kwargs"])
    state = load_file(os.path.join(MODEL_PATHS[name], "model.safetensors"))
    result = model.load_state_dict(state)
    print(f"[{name}] load_state_dict: {result}")
    if hasattr(model, "lm_head"):
        del model.lm_head
    return model

for name in BACKBONE_REGISTRY:
    m = load_backbone(name)
    print(f"{name}: {sum(p.numel() for p in m.parameters())/1e6:.1f}M params")
    del m

## 14. Multi-label document classifier wrapper

In [ ]:
class MultiLabelDocClassifier(nn.Module):
    def __init__(self, lm, n_embd, n_labels, dropout=0.1):
        super().__init__()
        self.lm = lm
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * n_embd, n_embd), nn.GELU(approximate='tanh'),
            nn.Dropout(dropout), nn.Linear(n_embd, n_labels)
        )

    def forward(self, input_ids, attention_mask=None, labels=None, pos_weight=None, threshold_offset=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        x = self.lm.wte(input_ids) + self.lm.wpe(positions)
        x = self.lm.drop(x)

        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device)).view(1, 1, T, T)
        attn_mask = causal
        if attention_mask is not None:
            attn_mask = causal & attention_mask[:, None, None, :].bool()

        for block in self.lm.h:
            x, _ = block(x, attn_mask=attn_mask)
        x = self.lm.ln_f(x)

        if attention_mask is not None:
            last_idx = attention_mask.sum(dim=1) - 1
            m = attention_mask.unsqueeze(-1).float()
        else:
            last_idx = torch.full((B,), T - 1, device=input_ids.device)
            m = torch.ones(B, T, 1, device=input_ids.device)
        last = x[torch.arange(B, device=input_ids.device), last_idx]
        mean = ((x.float() * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)).to(x.dtype)
        pooled = torch.cat([last, mean], dim=-1)

        logits = self.head(self.dropout(pooled))
        if threshold_offset is not None:
            logits = logits + threshold_offset
        loss = None
        if labels is not None:
            loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)
        return {"loss": loss, "logits": logits}

print("MultiLabelDocClassifier defined.")

## 15. Metrics and threshold calibration

Multi-label macro-F1/P/R (averaged per statute), micro-F1, and subset accuracy (exact-set match).
Threshold tuning follows the same half/half dev-generalization check used in Phases 3-4.

In [ ]:
def multilabel_prf(y_true, y_pred):
    """y_true, y_pred: (N, N_STATUTES) binary arrays."""
    tp = (y_true * y_pred).sum(axis=0)
    pred_pos = y_pred.sum(axis=0)
    true_pos = y_true.sum(axis=0)
    P = np.divide(tp, pred_pos, out=np.zeros_like(tp, dtype=float), where=pred_pos > 0)
    R = np.divide(tp, true_pos, out=np.zeros_like(tp, dtype=float), where=true_pos > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros_like(P), where=(P + R) > 0)
    # statutes with zero true examples in this split are excluded from the macro average
    valid = true_pos > 0
    return P, R, F, valid


def micro_f1(y_true, y_pred):
    tp = (y_true * y_pred).sum()
    fp = ((1 - y_true) * y_pred).sum()
    fn = (y_true * (1 - y_pred)).sum()
    P = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    R = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * P * R / (P + R) if (P + R) > 0 else 0.0


def subset_accuracy(y_true, y_pred):
    return (y_true == y_pred).all(axis=1).mean()


def print_multilabel_scores(y_true, y_pred, title=""):
    print(f"--- {title} ---")
    P, R, F, valid = multilabel_prf(y_true, y_pred)
    macro_f1 = F[valid].mean() if valid.any() else 0.0
    print(f"MACRO (over {valid.sum()} statutes with >=1 dev example)  P {P[valid].mean():.4f}  R {R[valid].mean():.4f}  F1 {macro_f1:.4f}")
    print(f"Micro-F1: {micro_f1(y_true, y_pred):.4f}")
    print(f"Subset accuracy: {subset_accuracy(y_true, y_pred):.4f}")
    print(f"avg predicted statutes/doc: {y_pred.sum(axis=1).mean():.2f} | avg true statutes/doc: {y_true.sum(axis=1).mean():.2f}")
    print(f"docs with zero predicted statutes: {(y_pred.sum(axis=1) == 0).sum()} / {len(y_pred)}")
    return macro_f1


def raw_macro_f1_multilabel(logits, labels, threshold=0.5):
    preds = (torch.sigmoid(logits) >= threshold).numpy().astype(int)
    y_true = labels.numpy().astype(int)
    P, R, F, valid = multilabel_prf(y_true, preds)
    return F[valid].mean() if valid.any() else 0.0


@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
        all_logits.append(out["logits"].float().cpu())
        if "labels" in b:
            all_labels.append(b["labels"].cpu())
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels) if all_labels else None
    return logits, labels


def tune_threshold(logits, labels, n_steps=37):
    probs = torch.sigmoid(logits).numpy()
    y_true = labels.numpy().astype(int)
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.05, 0.95, n_steps):
        preds = (probs >= t).astype(int)
        P, R, F, valid = multilabel_prf(y_true, preds)
        f1 = F[valid].mean() if valid.any() else 0.0
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    return best_t, best_f1


def threshold_calibration_check(logits, labels, min_gain=0.005):
    n = len(labels)
    idx = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
    half = n // 2
    a, b = idx[:half], idx[half:]

    t_a, _ = tune_threshold(logits[a], labels[a])
    probs_b = torch.sigmoid(logits[b]).numpy()
    y_b = labels[b].numpy().astype(int)
    preds_b_with = (probs_b >= t_a).astype(int)
    preds_b_without = (probs_b >= 0.5).astype(int)
    Pb, Rb, Fb, vb = multilabel_prf(y_b, preds_b_with)
    f1_b_with = Fb[vb].mean() if vb.any() else 0.0
    Pb0, Rb0, Fb0, vb0 = multilabel_prf(y_b, preds_b_without)
    f1_b_without = Fb0[vb0].mean() if vb0.any() else 0.0

    t_b, _ = tune_threshold(logits[b], labels[b])
    probs_a = torch.sigmoid(logits[a]).numpy()
    y_a = labels[a].numpy().astype(int)
    preds_a_with = (probs_a >= t_b).astype(int)
    preds_a_without = (probs_a >= 0.5).astype(int)
    Pa, Ra, Fa, va = multilabel_prf(y_a, preds_a_with)
    f1_a_with = Fa[va].mean() if va.any() else 0.0
    Pa0, Ra0, Fa0, va0 = multilabel_prf(y_a, preds_a_without)
    f1_a_without = Fa0[va0].mean() if va0.any() else 0.0

    gain = ((f1_b_with - f1_b_without) + (f1_a_with - f1_a_without)) / 2
    print(f"held-out half: F1 at 0.5 {f1_b_without:.4f} | at tuned threshold {f1_b_with:.4f}")
    print(f"held-out half: F1 at 0.5 {f1_a_without:.4f} | at tuned threshold {f1_a_with:.4f}")
    print(f"mean held-out gain from threshold tuning: {gain:+.4f}")
    return gain

print("Metrics and calibration utilities defined.")

## 16. Submission writer

List of `{"id", "pred": [int, ...]}`, unique statute IDs per document, mapped back from internal
label indices via `ID2STATUTE`.

In [ ]:
def write_submission(paths, model, threshold=0.5):
    test_ds = LSIDataset(test_records, with_labels=False)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=lsi_collator)

    model.eval()
    all_logits = []
    with torch.no_grad():
        for b in test_loader:
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            all_logits.append(out["logits"].float().cpu())
    logits = torch.cat(all_logits)
    preds = (torch.sigmoid(logits) >= threshold).numpy().astype(int)

    prediction = []
    for r, row in zip(test_records, preds):
        statute_ids = sorted({int(ID2STATUTE[i]) for i in np.where(row == 1)[0]})
        prediction.append({"id": r["id"], "pred": statute_ids})

    assert len(prediction) == len(test_records)
    assert all(len(p["pred"]) == len(set(p["pred"])) for p in prediction)
    assert {p["id"] for p in prediction} == {r["id"] for r in test_records}

    with open(paths["prediction_path"], "w", encoding="utf-8") as f:
        json.dump(prediction, f, ensure_ascii=False)
    with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(paths["prediction_path"], arcname="prediction.json")
    print("saved:", paths["submission_zip"])
    with zipfile.ZipFile(paths["submission_zip"]) as z:
        print(z.namelist())
    return prediction

print("Submission writer defined.")

## 17. Generic training loop

In [ ]:
def train_classifier(model, optimizer, scheduler, train_loader, dev_loader,
                     pos_weight, epochs, patience, paths, grad_accum=1, min_cal_gain=0.005,
                     final_dev_loader=None):
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = {"train": [], "dev_f1": []}
    best_f1, bad_epochs = 0.0, 0
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        run, cnt = 0.0, 0
        optimizer.zero_grad()
        for i, b in enumerate(train_loader):
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(**b, pos_weight=pos_weight)
            scaler.scale(out["loss"] / grad_accum).backward()
            run += out["loss"].item(); cnt += 1
            if (i + 1) % grad_accum == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad()
                if scheduler is not None:
                    scheduler.step()

        logits, labels = get_logits(model, dev_loader)
        f1 = raw_macro_f1_multilabel(logits, labels)
        history["train"].append(run / cnt); history["dev_f1"].append(f1)
        print(f"epoch {epoch+1}/{epochs} | train loss {run/cnt:.4f} | dev macro-F1 (uncalibrated, threshold 0.5, dev subset) {f1:.4f} | {time.time()-t0:.0f}s")

        if f1 > best_f1:
            best_f1, bad_epochs = f1, 0
            torch.save(model.state_dict(), paths["model_path"])
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"early stopping at epoch {epoch+1} (best uncalibrated dev macro-F1 {best_f1:.4f})")
                break

    model.load_state_dict(torch.load(paths["model_path"], weights_only=True))

    eval_loader = final_dev_loader if final_dev_loader is not None else dev_loader
    logits, labels = get_logits(model, eval_loader)

    print("threshold calibration check:")
    gain = threshold_calibration_check(logits, labels, min_cal_gain)
    if gain > min_cal_gain:
        threshold, _ = tune_threshold(logits, labels)
        title = "FINAL (best checkpoint, threshold calibrated)"
    else:
        threshold = 0.5
        title = "FINAL (best checkpoint, default threshold 0.5)"
        print("threshold not adjusted")

    preds = (torch.sigmoid(logits) >= threshold).numpy().astype(int)
    final_f1 = print_multilabel_scores(labels.numpy().astype(int), preds, title=title)
    print("best uncalibrated dev macro-F1:", best_f1, "| final reported:", float(final_f1), "| threshold:", threshold)

    return model, history, threshold, float(best_f1), float(final_f1)

print("train_classifier defined.")

---
## Experiment 1 -- GQA + LoRMA

In [ ]:
class LoRMALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base_linear.out_features
        self.r = r; self.alpha = alpha
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.A = nn.Parameter(torch.randn(r, d_out) * 0.01)
        self.register_buffer("eye", torch.eye(d_out))

    def forward(self, x):
        BA = self.B @ self.A
        M = self.eye + (self.alpha / self.r) * BA
        W_eff = M @ self.base.weight
        return F.linear(x, W_eff, self.base.bias)


def apply_lorma(model, target_names, r=8, alpha=16):
    for parent_name, parent in model.named_modules():
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                setattr(parent, child_name, LoRMALinear(child, r=r, alpha=alpha))
    return model

print("LoRMA defined.")

In [ ]:
EXP1_NAME = "01_gqa_lorma"
paths = experiment_dirs(EXP1_NAME)

LORMA_R, LORMA_ALPHA = 8, 16
LORMA_TARGETS = ["q_proj", "v_proj"]
EPOCHS, BATCH_SIZE, GRAD_ACCUM, PATIENCE = 15, 8, 2, 7
LR_LORMA, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO = 1e-4, 1e-4, 0.05, 0.10

backbone = load_backbone("gqa")
for p in backbone.parameters():
    p.requires_grad = False
backbone = apply_lorma(backbone, LORMA_TARGETS, r=LORMA_R, alpha=LORMA_ALPHA)
model = MultiLabelDocClassifier(backbone, n_embd=N_EMBD, n_labels=N_STATUTES).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(batch_size=8)

lorma_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": lorma_params, "lr": LR_LORMA},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, threshold, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    pos_weight, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP1_NAME, "phase1_model": "GQA", "technique": "LoRMA",
    "r": LORMA_R, "alpha": LORMA_ALPHA, "targets": LORMA_TARGETS,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS, "n_statutes": N_STATUTES,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold": threshold,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1 (dev subset)",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO = globals().get("EXPERIMENT_RUN_INFO", {})
EXPERIMENT_RUN_INFO[EXP1_NAME] = run_info
globals()["EXPERIMENT_RUN_INFO"] = EXPERIMENT_RUN_INFO

prediction = write_submission(paths, model, threshold)

_exp1_paths = experiment_dirs(EXP1_NAME)
def build_exp1_trained():
    bb = load_backbone("gqa")
    for p in bb.parameters(): p.requires_grad = False
    bb = apply_lorma(bb, LORMA_TARGETS, r=LORMA_R, alpha=LORMA_ALPHA)
    m = MultiLabelDocClassifier(bb, N_EMBD, N_STATUTES).to(device)
    m.load_state_dict(torch.load(_exp1_paths["model_path"], weights_only=True))
    m.eval(); return m

EXPERIMENT_BUILDERS = globals().get("EXPERIMENT_BUILDERS", {})
EXPERIMENT_BUILDERS[EXP1_NAME] = build_exp1_trained
globals()["EXPERIMENT_BUILDERS"] = EXPERIMENT_BUILDERS
print("registered:", EXP1_NAME)

---
## Experiment 2 -- GPT-2 + Full Fine-tuning

In [ ]:
EXP2_NAME = "02_gpt2_fullfinetune"
paths = experiment_dirs(EXP2_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM, PATIENCE = 15, 8, 2, 7
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO = 3e-5, 1e-3, 0.05, 0.10

backbone = load_backbone("gpt2")
for p in backbone.parameters():
    p.requires_grad = True
model = MultiLabelDocClassifier(backbone, n_embd=N_EMBD, n_labels=N_STATUTES).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(batch_size=8)

backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
head_params = [p for n, p in model.named_parameters() if "head" in n]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, threshold, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    pos_weight, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP2_NAME, "phase1_model": "GPT-2", "technique": "Full Fine-tuning",
    "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS, "n_statutes": N_STATUTES,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold": threshold,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1 (dev subset)",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP2_NAME] = run_info
prediction = write_submission(paths, model, threshold)

_exp2_paths = experiment_dirs(EXP2_NAME)
def build_exp2_trained():
    bb = load_backbone("gpt2")
    m = MultiLabelDocClassifier(bb, N_EMBD, N_STATUTES).to(device)
    m.load_state_dict(torch.load(_exp2_paths["model_path"], weights_only=True))
    m.eval(); return m
EXPERIMENT_BUILDERS[EXP2_NAME] = build_exp2_trained
print("registered:", EXP2_NAME)

---
## Experiment 3 -- SwiGLU + AdaLoRA

Adaptive rank allocation via `peft.AdaLoraConfig`, attached with `inject_adapter_in_model` per
your TA's instruction for custom (non-`PreTrainedModel`) architectures.

**Caveat, stated upfront:** AdaLoRA's rank-budget scheduling normally calls
`model.update_and_allocate(step)` each step (needs the full `PeftModel` wrapper from
`get_peft_model`, not guaranteed available via the lower-level injection API). The training loop
below tries that call and prints whether it actually fired; if it silently can't, AdaLoRA still
trains correctly as a fixed-rank LoRA variant with rank-importance regularization active, just
without the rank *reallocation* schedule -- noted honestly in `run_info`, not hidden.

In [ ]:
from peft import AdaLoraConfig, inject_adapter_in_model

def build_backbone_swiglu_adalora(total_steps):
    bb = load_backbone("swiglu")
    for p in bb.parameters():
        p.requires_grad = False
    adalora_config = AdaLoraConfig(
        init_r=12, target_r=8, r=12, lora_alpha=16,
        target_modules=["w1", "w3"], lora_dropout=0.05,
        tinit=max(1, total_steps // 10), tfinal=max(2, total_steps // 5),
        deltaT=max(1, total_steps // 50), total_step=total_steps,
    )
    bb = inject_adapter_in_model(adalora_config, bb)
    return bb, adalora_config

print("AdaLoRA builder defained.")


In [ ]:
EXP3_NAME = "03_swiglu_adalora"
paths = experiment_dirs(EXP3_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM, PATIENCE = 15, 8, 2, 7
LR_ADAPTER, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO = 1e-3, 1e-3, 0.05, 0.10

_tmp_train_loader = make_train_loader(BATCH_SIZE)
_total_steps_estimate = math.ceil(len(_tmp_train_loader) / GRAD_ACCUM) * EPOCHS

backbone, adalora_config = build_backbone_swiglu_adalora(_total_steps_estimate)
model = MultiLabelDocClassifier(backbone, n_embd=N_EMBD, n_labels=N_STATUTES).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
has_update_and_allocate = hasattr(backbone, "update_and_allocate")
print("backbone has update_and_allocate (rank reallocation available):", has_update_and_allocate)

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(batch_size=8)

adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, threshold, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    pos_weight, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP3_NAME, "phase1_model": "SwiGLU", "technique": "AdaLoRA",
    "rank_reallocation_available": has_update_and_allocate,
    "lr_adapter": LR_ADAPTER, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS, "n_statutes": N_STATUTES,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold": threshold,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1 (dev subset)",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP3_NAME] = run_info
prediction = write_submission(paths, model, threshold)

_exp3_paths = experiment_dirs(EXP3_NAME)
_exp3_total_steps = _total_steps_estimate
def build_exp3_trained():
    bb, _ = build_backbone_swiglu_adalora(_exp3_total_steps)
    m = MultiLabelDocClassifier(bb, N_EMBD, N_STATUTES).to(device)
    m.load_state_dict(torch.load(_exp3_paths["model_path"], weights_only=True))
    m.eval(); return m
EXPERIMENT_BUILDERS[EXP3_NAME] = build_exp3_trained
print("registered:", EXP3_NAME)

---
## Experiment 4 -- Prefix-LM + Adapters

In [ ]:
class AdapterBottleneck(nn.Module):
    def __init__(self, n_embd, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(n_embd, bottleneck)
        self.act = nn.GELU(approximate='tanh')
        self.up = nn.Linear(bottleneck, n_embd)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


class AdapterBlockWrapper(nn.Module):
    def __init__(self, block, n_embd, bottleneck=64):
        super().__init__()
        self.block = block
        for p in self.block.parameters():
            p.requires_grad = False
        self.adapter = AdapterBottleneck(n_embd, bottleneck)

    def forward(self, x, layer_past=None, attn_mask=None):
        x, present = self.block(x, layer_past=layer_past, attn_mask=attn_mask)
        x = self.adapter(x)
        return x, present


def apply_adapters(model, n_embd, bottleneck=64):
    for i in range(len(model.h)):
        model.h[i] = AdapterBlockWrapper(model.h[i], n_embd, bottleneck)
    return model

print("Adapters defined.")

In [ ]:
EXP4_NAME = "04_prefixlm_adapters"
paths = experiment_dirs(EXP4_NAME)

ADAPTER_BOTTLENECK = 64
EPOCHS, BATCH_SIZE, GRAD_ACCUM, PATIENCE = 15, 8, 2, 7
LR_ADAPTER, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO = 1e-3, 1e-3, 0.05, 0.10

backbone = load_backbone("prefixlm")
for p in backbone.parameters():
    p.requires_grad = False
backbone = apply_adapters(backbone, N_EMBD, bottleneck=ADAPTER_BOTTLENECK)
model = MultiLabelDocClassifier(backbone, n_embd=N_EMBD, n_labels=N_STATUTES).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(batch_size=8)

adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, threshold, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    pos_weight, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP4_NAME, "phase1_model": "Prefix-LM", "technique": "Adapters (Houlsby bottleneck)",
    "bottleneck_dim": ADAPTER_BOTTLENECK, "lr_adapter": LR_ADAPTER, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS, "n_statutes": N_STATUTES,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold": threshold,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1 (dev subset)",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP4_NAME] = run_info
prediction = write_submission(paths, model, threshold)

_exp4_paths = experiment_dirs(EXP4_NAME)
def build_exp4_trained():
    bb = load_backbone("prefixlm")
    for p in bb.parameters(): p.requires_grad = False
    bb = apply_adapters(bb, N_EMBD, bottleneck=ADAPTER_BOTTLENECK)
    m = MultiLabelDocClassifier(bb, N_EMBD, N_STATUTES).to(device)
    m.load_state_dict(torch.load(_exp4_paths["model_path"], weights_only=True))
    m.eval(); return m
EXPERIMENT_BUILDERS[EXP4_NAME] = build_exp4_trained
print("registered:", EXP4_NAME)

---
## Experiment 5 -- GQA + Prefix Tuning

Hand-rolled. Per-layer learnable K/V vectors (`n_prefix=20` virtual tokens), injected directly
via each block's existing `layer_past` mechanism -- the same mechanism these blocks already use
for incremental decoding, repurposed here. Genuinely different from Experiment 5's analogue in
Phase 3 (Prompt Tuning), which only prepended learned *input embeddings*.

In [ ]:
N_PREFIX = 20

class PrefixTuningClassifier(nn.Module):
    def __init__(self, lm, n_embd, n_head, n_kv_head, n_layer, n_labels, n_prefix=N_PREFIX, dropout=0.1):
        super().__init__()
        self.lm = lm
        for p in self.lm.parameters():
            p.requires_grad = False
        self.n_prefix = n_prefix
        self.head_dim = n_embd // n_head
        self.prefix_k = nn.Parameter(torch.randn(n_layer, n_kv_head, n_prefix, self.head_dim) * 0.02)
        self.prefix_v = nn.Parameter(torch.randn(n_layer, n_kv_head, n_prefix, self.head_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * n_embd, n_embd), nn.GELU(approximate='tanh'),
            nn.Dropout(dropout), nn.Linear(n_embd, n_labels)
        )

    def forward(self, input_ids, attention_mask=None, labels=None, pos_weight=None, threshold_offset=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        x = self.lm.wte(input_ids) + self.lm.wpe(positions)
        x = self.lm.drop(x)

        seq_len = self.n_prefix + T
        prefix_visible = torch.ones(B, self.n_prefix, dtype=torch.bool, device=input_ids.device)
        key_visible = torch.cat([prefix_visible, attention_mask], dim=1) if attention_mask is not None \
            else torch.ones(B, seq_len, dtype=torch.bool, device=input_ids.device)
        causal_part = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
        prefix_part = torch.ones(T, self.n_prefix, dtype=torch.bool, device=input_ids.device)
        causal_full = torch.cat([prefix_part, causal_part], dim=1)        # (T, n_prefix+T)
        attn_mask = causal_full.view(1, 1, T, seq_len) & key_visible[:, None, None, :]

        for i, block in enumerate(self.lm.h):
            pk = self.prefix_k[i].unsqueeze(0).expand(B, -1, -1, -1).to(x.dtype)
            pv = self.prefix_v[i].unsqueeze(0).expand(B, -1, -1, -1).to(x.dtype)
            x, _ = block(x, layer_past=(pk, pv), attn_mask=attn_mask)
        x = self.lm.ln_f(x)

        if attention_mask is not None:
            last_idx = attention_mask.sum(dim=1) - 1
            m = attention_mask.unsqueeze(-1).float()
        else:
            last_idx = torch.full((B,), T - 1, device=input_ids.device)
            m = torch.ones(B, T, 1, device=input_ids.device)
        last = x[torch.arange(B, device=input_ids.device), last_idx]
        mean = ((x.float() * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)).to(x.dtype)
        pooled = torch.cat([last, mean], dim=-1)

        logits = self.head(self.dropout(pooled))
        if threshold_offset is not None:
            logits = logits + threshold_offset
        loss = None
        if labels is not None:
            loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)
        return {"loss": loss, "logits": logits}

print("Prefix Tuning classifier defined.")

In [ ]:
EXP5_NAME = "05_gqa_prefixtuning"
paths = experiment_dirs(EXP5_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM, PATIENCE = 15, 8, 2, 7
LR_PREFIX, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO = 1e-2, 1e-3, 0.05, 0.10

backbone = load_backbone("gqa")
model = PrefixTuningClassifier(backbone, n_embd=N_EMBD, n_head=N_HEAD, n_kv_head=N_KV_HEAD,
                               n_layer=N_LAYER, n_labels=N_STATUTES, n_prefix=N_PREFIX).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(batch_size=8)

prefix_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": prefix_params, "lr": LR_PREFIX},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, threshold, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    pos_weight, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP5_NAME, "phase1_model": "GQA", "technique": "Prefix Tuning (hand-rolled)",
    "n_prefix": N_PREFIX, "lr_prefix": LR_PREFIX, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS, "n_statutes": N_STATUTES,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold": threshold,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1 (dev subset)",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP5_NAME] = run_info
prediction = write_submission(paths, model, threshold)

_exp5_paths = experiment_dirs(EXP5_NAME)
def build_exp5_trained():
    bb = load_backbone("gqa")
    m = PrefixTuningClassifier(bb, N_EMBD, N_HEAD, N_KV_HEAD, N_LAYER, N_STATUTES, n_prefix=N_PREFIX).to(device)
    m.load_state_dict(torch.load(_exp5_paths["model_path"], weights_only=True))
    m.eval(); return m
EXPERIMENT_BUILDERS[EXP5_NAME] = build_exp5_trained
print("registered:", EXP5_NAME)

---
## Pick the winner of Experiments 1-5

In [ ]:
WINNER_CANDIDATES = ["01_gqa_lorma", "02_gpt2_fullfinetune", "03_swiglu_adalora",
                    "04_prefixlm_adapters", "05_gqa_prefixtuning"]
scored = [(EXPERIMENT_RUN_INFO[n]["final_dev_f1"], n) for n in WINNER_CANDIDATES if n in EXPERIMENT_RUN_INFO]
scored.sort(reverse=True)
print("candidates (sorted by final dev macro-F1):")
for f1, n in scored:
    print(f"  {n:30s} {f1:.4f}")
WINNER_NAME = scored[0][1]
print("\nWinner:", WINNER_NAME)

BACKBONE_OF = {
    "01_gqa_lorma": "gqa", "02_gpt2_fullfinetune": "gpt2", "03_swiglu_adalora": "swiglu",
    "04_prefixlm_adapters": "prefixlm", "05_gqa_prefixtuning": "gqa",
}
WINNER_BACKBONE = BACKBONE_OF[WINNER_NAME]
print("winner backbone family:", WINNER_BACKBONE)

---
## Experiment 6 -- Knowledge Distillation

Teacher = winner above, frozen. Student = half the layers (6 of 12) of the **same architecture
family**, initialized from the teacher's own **Phase 1** pretrained weights (not its fine-tuned
adapter), then trained fully with a combined hard-label + soft-teacher-matching BCE loss --
the multi-label analogue of Phase 3's KD (soft targets are the teacher's temperature-scaled
sigmoid probabilities, matched via BCE rather than a softmax KL, since labels aren't mutually
exclusive here).

In [ ]:
STUDENT_N_LAYER = N_LAYER // 2

def build_student_backbone(backbone_name):
    spec = BACKBONE_REGISTRY[backbone_name]
    kwargs = dict(spec["kwargs"]); kwargs["n_layer"] = STUDENT_N_LAYER
    student = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **kwargs)
    full_state = load_file(os.path.join(MODEL_PATHS[backbone_name], "model.safetensors"))
    student_state = student.state_dict()
    for k in list(student_state.keys()):
        if k.startswith("h."):
            layer_idx = int(k.split(".")[1])
            if layer_idx >= STUDENT_N_LAYER:
                continue
        if k in full_state and full_state[k].shape == student_state[k].shape:
            student_state[k] = full_state[k]
    student.load_state_dict(student_state)
    if hasattr(student, "lm_head"):
        del student.lm_head
    return student


def distillation_loss_multilabel(student_logits, teacher_logits, labels, pos_weight, T=2.0, alpha=0.5):
    hard = F.binary_cross_entropy_with_logits(student_logits, labels, pos_weight=pos_weight)
    teacher_probs = torch.sigmoid(teacher_logits / T)
    soft = F.binary_cross_entropy_with_logits(student_logits / T, teacher_probs) * (T ** 2)
    return alpha * hard + (1 - alpha) * soft

print("KD helpers defined. Student layers:", STUDENT_N_LAYER, "of", N_LAYER)

In [ ]:
EXP6_NAME = "06_knowledge_distillation"
paths = experiment_dirs(EXP6_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM, PATIENCE = 15, 8, 2, 7
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO = 3e-5, 1e-3, 0.05, 0.10
KD_TEMPERATURE, KD_ALPHA = 2.0, 0.5

teacher_model = EXPERIMENT_BUILDERS[WINNER_NAME]()
for p in teacher_model.parameters():
    p.requires_grad = False

student_backbone = build_student_backbone(WINNER_BACKBONE)
model = MultiLabelDocClassifier(student_backbone, n_embd=N_EMBD, n_labels=N_STATUTES).to(device)
for p in model.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
teacher_total = sum(p.numel() for p in teacher_model.parameters())
print(f"student trainable: {trainable:,} ({total:,} total) | teacher: {teacher_total:,}")

train_loader = make_train_loader(BATCH_SIZE)
dev_loader = make_dev_loader(batch_size=8)

backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
head_params = [p for n, p in model.named_parameters() if "head" in n]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
history = {"train": [], "dev_f1": []}
best_f1, bad_epochs = 0.0, 0
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    run, cnt = 0.0, 0
    optimizer.zero_grad()
    for i, b in enumerate(train_loader):
        b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            teacher_logits = teacher_model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])["logits"]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            student_out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            loss = distillation_loss_multilabel(student_out["logits"], teacher_logits, b["labels"],
                                                pos_weight, T=KD_TEMPERATURE, alpha=KD_ALPHA)
        scaler.scale(loss / GRAD_ACCUM).backward()
        run += loss.item(); cnt += 1
        if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

    logits, labels = get_logits(model, dev_loader)
    f1 = raw_macro_f1_multilabel(logits, labels)
    history["train"].append(run / cnt); history["dev_f1"].append(f1)
    print(f"epoch {epoch+1}/{EPOCHS} | train loss {run/cnt:.4f} | dev macro-F1 (uncalibrated) {f1:.4f} | {time.time()-t0:.0f}s")

    if f1 > best_f1:
        best_f1, bad_epochs = f1, 0
        torch.save(model.state_dict(), paths["model_path"])
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch+1} (best uncalibrated dev macro-F1 {best_f1:.4f})")
            break

model.load_state_dict(torch.load(paths["model_path"], weights_only=True))
logits, labels = get_logits(model, dev_loader)
print("threshold calibration check:")
gain = threshold_calibration_check(logits, labels)
if gain > 0.005:
    threshold, _ = tune_threshold(logits, labels)
else:
    threshold = 0.5
    print("threshold not adjusted")
preds = (torch.sigmoid(logits) >= threshold).numpy().astype(int)
final_f1 = print_multilabel_scores(labels.numpy().astype(int), preds, title="FINAL (student, KD)")

teacher_final_f1 = EXPERIMENT_RUN_INFO[WINNER_NAME]["final_dev_f1"]
print(f"\nteacher ({WINNER_NAME}) final dev F1: {teacher_final_f1:.4f} | student (KD) final dev F1: {final_f1:.4f}")

In [ ]:
run_info = {
    "experiment": EXP6_NAME, "technique": "Knowledge Distillation",
    "teacher": WINNER_NAME, "teacher_backbone": WINNER_BACKBONE, "teacher_final_dev_f1": teacher_final_f1,
    "student_backbone": WINNER_BACKBONE, "student_n_layer": STUDENT_N_LAYER,
    "kd_temperature": KD_TEMPERATURE, "kd_alpha": KD_ALPHA,
    "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS, "n_statutes": N_STATUTES,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "teacher_params": teacher_total,
    "threshold": threshold, "pooling": "last+mean",
    "selection_metric": "uncalibrated dev macro-F1 (dev subset)",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP6_NAME] = run_info
prediction = write_submission(paths, model, threshold)

---
## Summary -- all 6 submissions

In [ ]:
import glob
print(f"{'experiment':30s} {'final_dev_f1':>13s}")
for d in sorted(glob.glob(os.path.join(BASE_OUTPUT_DIR, "*", "run_info.json"))):
    with open(d) as f:
        ri = json.load(f)
    f1 = ri.get("final_dev_f1", ri.get("best_dev_f1", float("nan")))
    print(f"{ri['experiment']:30s} {f1:13.4f}")